# 01 · EDA: Clientes y Red Comercial

**Prueba analítica DICAGI 2022 — Modelo de Capacidad · Bancolombia**

---

## 📌 Objetivo del notebook

Entender **quiénes son los clientes** del segmento Preferencial y **cómo está organizada la red comercial** que los atiende, para tomar decisiones informadas sobre cómo modelar la asignación.

## 🧠 Mindset

> *"Nunca optimices lo que no entiendes."*

Antes de escribir una sola línea de modelo matemático, necesitamos responder cinco preguntas:
1. **¿Cuántos** clientes hay y cómo se reparten en categorías A / B / C?
2. **¿Cómo se ve el score** y si discrimina A > B > C?
3. **¿Dónde** están geográficamente?
4. **¿Cómo** está organizada la red (clientes por ejecutivo, ejecutivos por gerente)?
5. **¿Cierran las llaves** entre tablas? (cuántos clientes son realmente asignables)

## 🎓 Cómo leer este notebook (tres capas)

Cada sección importante incluye:

| Capa | Para quién |
|---|---|
| 👥 **Versión simple** | Público no técnico |
| 🔬 **Perfil técnico** | Data scientist / analista |
| ⚛️ **Análogo físico** | Intuición desde física estadística |

## Tablas usadas

| Tabla | Filas | Rol |
|---|---|---|
| `pcac_mac_gpi_clientes.csv` | 34,145 | Población de clientes con categoría, score y asignación actual |
| `pcac_mac_gpi_ecas.csv` | 392 | Relación gerente ↔ ejecutivo (estructura de la red) |
| `pcac_planta_comercial2.csv` | 50 | Catálogo de gerentes con ciudad/región/banca |

---
## 0. Setup del entorno

### 🎯 ¿Qué hacemos?
Cargamos librerías, hacemos `src/` importable y activamos el tema visual corporativo de Bancolombia.

### 🧠 ¿Por qué?
- **IDs como string**: los identificadores son enteros de 19+ dígitos. `int64` solo soporta hasta 19 dígitos exactos antes de perder precisión. Cargarlos como string evita pérdida silenciosa que solo se notaría al fallar un `merge` mucho más adelante.
- **Tema Plotly Bancolombia**: registrarlo aquí garantiza que **todas** las gráficas salen con la identidad corporativa, sin re-especificar colores en cada `figure`.

### ⚛️ Análogo físico
Equivale a **calibrar los instrumentos antes de medir**: si la balanza tiene resolución 0.1g no puedes medir efectos de 0.01g. Si el tipo de dato no preserva precisión, todos los joins y conteos están condenados a errores.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from modelo_capacidad.data.loader import load_clientes, load_ecas, load_planta
from modelo_capacidad.viz import apply_theme, BANCOLOMBIA_COLORS, BANCOLOMBIA_SEQUENTIAL

apply_theme()
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)
print('Entorno listo. Tema Bancolombia activo en Plotly.')

---
## 1. Carga e inspección rápida

### 🎯 ¿Qué hacemos?
Cargamos las 3 tablas y verificamos su tamaño y estructura.

### 🧠 Mindset
*"Conoce la geografía antes de cruzar el río."* — sin saber el orden de magnitud no puedes elegir el método correcto. 34k clientes × 50 gerentes es manejable con MILP exacto; 34M no.

### 👥 Versión simple
Como llegar a una bodega nueva: cuentas las cajas, miras qué dice cada etiqueta, y abres una al azar para entender el formato. **No es tiempo perdido — es lo que evita errores costosos después.**

### 🔬 Versión técnica
- `.shape`, `.head()` y `.isna().sum()` son la rutina mínima. Cualquier `groupby` o `merge` sobre columna con muchos `NaN` puede dar resultados engañosos.

### ⚛️ Análogo físico
Es el **conteo de partículas** del sistema: $N_{clientes} \sim 10^4$, $N_{ejecutivos} \sim 10^2$, $N_{gerentes} \sim 10^1$. La **separación de escalas** sugiere aproximaciones jerárquicas (mean-field por nivel).

In [ ]:
clientes = load_clientes()
ecas = load_ecas()
planta = load_planta()

print(f'clientes: {clientes.shape[0]:>6,} filas × {clientes.shape[1]:>2} columnas')
print(f'ecas:     {ecas.shape[0]:>6,} filas × {ecas.shape[1]:>2} columnas')
print(f'planta:   {planta.shape[0]:>6,} filas × {planta.shape[1]:>2} columnas')
clientes.head(3)

In [ ]:
# Calidad: nulos por columna en clientes (top 15)
nulos = clientes.isna().sum().sort_values(ascending=False)
nulos = nulos[nulos > 0].head(15).to_frame('n_nulos')
nulos['pct'] = (nulos['n_nulos'] / len(clientes) * 100).round(2)
nulos

### 📏 Lectura de los datos reales

**Lo que vemos en esta corrida:**
- **34,145 clientes**, **392 filas en ecas**, **50 gerentes**.
- `marca_lista_ctrl` tiene **97% de nulos** (33,353/34,145) → no es columna confiable, ignorar.
- `perfil_riesgo_val` tiene **23% nulos**, `cod_dane` y `ciudad` ~16-19% → secundarias para el modelo.
- **Ojo**: `nombre_ejec_bco`, `cod_suc_ejec_bco`, `region_ejec_bco_1` tienen 292 nulos cada una (~0.85%) — son las **mismas filas** (clientes sin ejecutivo registrado de forma completa) y no podrán asignarse en el modelo.

**Decisión:** las columnas críticas para el modelo (`marca_mac_inv`, `score_modelo`, `cod_region_gte_inv`, `cod_ejec_bco`) están casi completas. Trabajaremos con ellas con confianza.

---
## 2. Distribución por categoría comercial (A / B / C)

### 🎯 ¿Qué hacemos?
Contamos cuántos clientes hay en cada categoría.

### 🧠 ¿Por qué este paso es CRÍTICO?
La métrica oficial de evaluación es:
$$\text{score} = \frac{\#A + \#B \text{ asignados}}{\#\text{ clientes totales}}$$

El **techo absoluto** de la métrica está dado por la mezcla A+B. Si el 40% son A+B, ningún algoritmo puede pasar de 0.40.

### 👥 Versión simple
Imagina que te pagan $1000 por vender a tipo A, $100 por B, $1 por C. Saber cuántos hay de cada tipo te dice el techo de comisiones posibles, **antes** de pensar en estrategia.

### 🔬 Versión técnica
Distribución empírica $\hat{p}(\text{cat})$ con `value_counts()`. 

**Alternativas que NO usamos:** *Pie chart* (estéticamente popular pero peor que barras para comparar magnitudes); *treemap* (overkill con 3 categorías).

### ⚛️ Análogo físico
Es la **distribución de niveles de energía**: cada cliente está en un "estado" (A, B, C) y la población de cada nivel determina, vía suma estadística, el valor objetivo total.

In [ ]:
cat = (clientes['marca_mac_inv']
       .value_counts(dropna=False)
       .rename_axis('categoria')
       .reset_index(name='clientes'))
cat['pct'] = (cat['clientes'] / cat['clientes'].sum() * 100).round(1)

fig = px.bar(
    cat, x='categoria', y='clientes', text='pct',
    title='Distribución de clientes por categoría comercial',
    color='categoria',
    color_discrete_map={'A': BANCOLOMBIA_COLORS['negro'],
                        'B': BANCOLOMBIA_COLORS['amarillo'],
                        'C': BANCOLOMBIA_COLORS['gris']},
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(showlegend=False, yaxis_title='# clientes', xaxis_title='Categoría')
fig.show()

# Techo teórico de la métrica oficial
pct_AB = cat.set_index('categoria').loc[['A', 'B'], 'pct'].sum()
print(f'\nTECHO ABSOLUTO de la métrica oficial (si todos los A+B se asignaran): {pct_AB:.1f}%')
cat

### 📏 Resultados reales y su significado

| Categoría | Clientes | % | Lectura |
|---|---|---|---|
| **A** | 5,495 | **16.1%** | Escasos pero los más rentables — peso máximo |
| **B** | 12,727 | **37.3%** | Volumen medio, mayoría productiva |
| **C** | 15,923 | **46.6%** | Casi la mitad. La métrica los ignora. |

**🎯 Techo absoluto de la métrica = A + B = 53.4%.** Ese es el valor máximo posible si **todos** los A y B fueran asignables. El modelo será juzgado por qué tan cerca de 53.4% llega.

**Decisión de modelado:** los pesos $w_A : w_B : w_C = 1000 : 100 : 1$ alinean al optimizador con la métrica oficial. Como A es escaso (16%), cada A asignado vale **mucho** y el optimizador debe priorizarlos agresivamente.

---
## 3. Score por categoría — ¿es coherente la jerarquía?

### 🎯 ¿Qué hacemos?
Verificamos con boxplot que el `score_modelo` sea **monotónicamente decreciente** A → B → C.

### 🧠 ¿Por qué importa?
El PDF dice: *"se le debe asignar a los gerentes los ejecutivos con los scores de clientes más altos"*. Si score y categoría están perfectamente alineados, basta con la categoría. Si no, **necesitamos ambos** en la función objetivo.

### 👥 Versión simple
Tu colegio etiqueta a los alumnos como "excelente / bueno / regular" Y les da nota 0-100. Si los "excelentes" siempre tienen >90, la etiqueta basta. Si hay "buenos" con 95 (mejores que algunos excelentes), la nota aporta info extra.

### 🔬 Versión técnica
Boxplot: mediana (línea), IQR (caja), whiskers (rango sin outliers), puntos (outliers). Solapamiento entre cajas indica baja discriminación.

### ⚛️ Análogo físico
El score es una **coordenada continua** que refina la categoría discreta. Como tener niveles cuánticos discretos (A, B, C) con desdoblamiento fino (fine structure) dentro de cada nivel.

In [ ]:
fig = px.box(
    clientes.dropna(subset=['marca_mac_inv', 'score_modelo']),
    x='marca_mac_inv', y='score_modelo',
    color='marca_mac_inv',
    category_orders={'marca_mac_inv': ['A', 'B', 'C']},
    title='Score del modelo por categoría comercial',
    color_discrete_map={'A': BANCOLOMBIA_COLORS['negro'],
                        'B': BANCOLOMBIA_COLORS['amarillo'],
                        'C': BANCOLOMBIA_COLORS['gris']},
    points='outliers',
)
fig.update_layout(showlegend=False, xaxis_title='Categoría', yaxis_title='Score')
fig.show()

clientes.groupby('marca_mac_inv')['score_modelo'].describe().round(3)

### 📏 Resultados reales

| Categoría | Mediana | IQR | Min | Max |
|---|---|---|---|---|
| A | **0.32** | [0.25, 0.41] | 0.06 | 1.00 |
| B | **0.09** | [0.09, 0.16] | **−0.03** | 0.54 |
| C | **0.03** | [0.00, 0.03] | **−0.03** | 0.54 |

**Hallazgos clave:**
1. ✅ Las medianas se separan claramente: **A (0.32) ≫ B (0.09) ≫ C (0.03)** → la jerarquía A>B>C se confirma.
2. ⚠️ **El score puede ser negativo** (mín −0.03). Esto NO es un error: el score puede ser un residual log-odds o salida sin normalizar de un modelo. **Implicación**: si lo usamos como tie-breaker, los negativos restan valor — coherente comercialmente (cliente indeseable resta).
3. ⚠️ **Outliers en C con score hasta 0.54** que superan la mediana de A. Hay clientes individualmente atractivos en C, pero la métrica oficial los descarta — la categoría manda.
4. **A muestra alta dispersión** (IQR ancho 0.16) → dentro de A hay subgrupos heterogéneos. El score es informativo como desempate dentro de A.

**Decisión de modelado:** $v_e = 1000 \cdot n^A_e + 100 \cdot n^B_e + 1 \cdot n^C_e + 0.1 \cdot \bar{s}_e$. Las categorías dominan, el score solo desempata.

---
## 4. Distribución geográfica — clientes por región × categoría

### 🎯 ¿Qué hacemos?
Tabla cruzada región × categoría visualizada como barras apiladas.

### 🧠 ¿Por qué importa?
**La restricción de zona es la más rígida del problema.** Un ejecutivo solo puede asignarse a un gerente de su misma región → el problema NO es uno solo, son **~5 sub-problemas independientes**.

Si una región tiene muchos A y pocos gerentes, ahí se concentra la escasez. Si otra tiene pocos clientes y muchos gerentes, la asignación es trivial allí.

### 👥 Versión simple
5 filiales de una pizzería en distintas ciudades, cada una con sus pizzeros y sus pedidos. **No puedes mandar un pizzero de Medellín a hacer un pedido en Cartagena.** Si Cartagena tiene 10× más pedidos que pizzeros, allá se pierden pedidos — aunque en Medellín sobren.

### 🔬 Versión técnica
Tabla de contingencia con `groupby`. Barras apiladas para magnitud total + mix. Alternativa: heatmap normalizado por fila para ver mix relativo.

### ⚛️ Análogo físico
**Principio de localidad**: el sistema no es un único reservorio mezclado, sino una **red de reservorios casi-independientes**. Análogo a tener bloques desconectados en spin glass — la energía total se descompone en suma de bloques, y cada bloque se resuelve por separado.

In [ ]:
zona_cat = (clientes
    .groupby(['region_gte_inv', 'marca_mac_inv'])
    .size()
    .reset_index(name='clientes'))

fig = px.bar(
    zona_cat, x='region_gte_inv', y='clientes', color='marca_mac_inv',
    barmode='stack',
    category_orders={'marca_mac_inv': ['A', 'B', 'C']},
    title='Clientes por región × categoría',
    color_discrete_map={'A': BANCOLOMBIA_COLORS['negro'],
                        'B': BANCOLOMBIA_COLORS['amarillo'],
                        'C': BANCOLOMBIA_COLORS['gris']},
)
fig.update_layout(xaxis_title='Región', yaxis_title='# clientes', legend_title='Categoría')
fig.show()

# Tabla con techo teórico por región
tabla = (zona_cat.pivot(index='region_gte_inv', columns='marca_mac_inv', values='clientes')
         .fillna(0).astype(int))
tabla['total'] = tabla.sum(axis=1)
tabla['pct_AB'] = ((tabla.get('A', 0) + tabla.get('B', 0)) / tabla['total'] * 100).round(1)
tabla.sort_values('total', ascending=False)

### 📏 Cómo leer la tabla

- **`total`**: tamaño del subproblema regional. Las regiones grandes dominan la métrica nacional (es un promedio ponderado por carga).
- **`pct_AB`**: techo de la métrica regional **si la capacidad fuera infinita**. 

**Diagnóstico rápido:** región con `pct_AB` alto + muchos gerentes = zona "fácil". Región con `pct_AB` bajo o pocos gerentes = el modelo tiene el techo bajo allí. La métrica nacional es el promedio ponderado de los `pct_AB` regionales (ponderado por capacidad efectiva).

---
## 5. Carga de clientes por ejecutivo — la unidad atómica

### 🎯 ¿Qué hacemos?
Histograma de cuántos clientes maneja cada ejecutivo.

### 🧠 ¿Por qué es CRUCIAL?
El PDF establece: *"todos los clientes del ejecutivo se asignan al gerente — es todo o nada"*. Esto significa que **la unidad de decisión NO es el cliente — es el ejecutivo**. Asignar un ejecutivo coloca 1, 50 o 200 clientes de golpe.

**Implicaciones:**
- Un ejecutivo con 200 clientes A es "oro": una decisión binaria coloca 200 A.
- Un ejecutivo con 5 clientes (todos C) es "penalizable": consume tiempo y aporta poco a la métrica.
- El problema cambia de tamaño: las binarias son **# ejecutivos**, no # clientes.

### 👥 Versión simple
Cada ejecutivo es una **caja sellada**. No decides cliente por cliente — decides la caja entera. Si la caja tiene clientes muy buenos y unos pocos malos, la abres todo. Si la caja es 99% mala, la dejas pasar aunque tenga un diamante adentro.

### 🔬 Versión técnica
Histograma + estadísticas descriptivas. Buscamos:
- **Distribución uniforme** → problema "limpio".
- **Cola larga (Pareto-like)** → típica en banca, complica el modelo (heterogeneidad fuerte).

### ⚛️ Análogo físico
Cada ejecutivo es una **partícula con masa variable**. Asignar = pegar la partícula al gerente. **Bin packing con piezas heterogéneas**: si las piezas son uniformes hay algoritmos triviales; si son heterogéneas, NP-hard.

In [ ]:
carga_ejec = (clientes
    .groupby('cod_ejec_bco')
    .agg(clientes=('num_doc_cli', 'count'),
         pct_A=('marca_mac_inv', lambda s: (s == 'A').mean() * 100),
         pct_B=('marca_mac_inv', lambda s: (s == 'B').mean() * 100),
         score_medio=('score_modelo', 'mean'))
    .reset_index())

fig = px.histogram(
    carga_ejec, x='clientes', nbins=40,
    title='Histograma · # clientes por ejecutivo',
    color_discrete_sequence=[BANCOLOMBIA_COLORS['amarillo']],
)
fig.update_layout(xaxis_title='# clientes que maneja el ejecutivo', yaxis_title='# ejecutivos', bargap=0.05)
fig.add_vline(x=carga_ejec['clientes'].median(), line_dash='dash',
              line_color=BANCOLOMBIA_COLORS['negro'],
              annotation_text=f'mediana = {int(carga_ejec["clientes"].median())}')
fig.show()

print('Estadísticas de carga por ejecutivo (TODOS los ejecutivos en clientes):')
carga_ejec['clientes'].describe().round(1)

### 📏 Resultados reales

**1,003 ejecutivos** únicos en `clientes`, distribución muy sesgada:

| Estadística | Valor | Lectura |
|---|---|---|
| Mediana | **8 clientes** | El ejecutivo "típico" lleva 8 clientes |
| Media | 34 | Media muy por encima de mediana → **cola larga** |
| P75 | 63 | El 25% superior maneja >63 clientes |
| Máximo | **284** | El "super-ejecutivo" maneja ~3.5% de los clientes solo él |
| P25 | 2 | El 25% inferior tiene 1-2 clientes (¿transición? ¿poco activos?) |

**Diagnóstico:**
- **Distribución de cola larga severa** (media/mediana = 4.25). Pocos ejecutivos manejan la mayor parte de la carga.
- **Heterogeneidad alta**: el rango va de 1 a 284. El greedy basado en `valor/tiempo` será efectivo aquí porque diferencia bien entre ejecutivos.
- **Riesgo del super-ejecutivo**: si el de 284 clientes no cabe en ningún gerente, perdemos esos 284 clientes de un golpe. Hay que verificar.

**🚨 Pero hay un problema más grave que se revela en la siguiente sección:** de estos **1,003 ejecutivos en `clientes`**, solo **392 están en `ecas`** (la tabla que define la jerarquía hacia gerente). Los otros 611 son potenciales huérfanos y debemos cuantificar cuántos clientes arrastran.

---
## 6. Validación de integridad referencial

### 🎯 ¿Qué hacemos?
Verificamos que los identificadores estén consistentes entre tablas. Y muy importante: **cuantificamos el impacto en clientes** de los huérfanos, no solo en ejecutivos.

### 🧠 ¿Por qué CRÍTICO?
Si descubrimos huérfanos al final, no sabemos si arrastran 100 clientes o 20,000. Cuantificar aquí define qué porcentaje de la población es **estructuralmente no asignable** vía nuestro modelo.

### 👥 Versión simple
Antes de armar un mueble verificas que **todos los tornillos del manual estén en la caja**. Pero **además** cuentas cuántas piezas dependen de los tornillos faltantes — no es lo mismo que falte un tornillo del estante (1 pieza afectada) a uno de la base (todo el mueble cae).

### 🔬 Versión técnica
Operaciones de conjuntos sobre IDs + agregación de impacto sobre `clientes`.

### ⚛️ Análogo físico
Es la **conservación de partículas**: si una tabla menciona un ID que la otra no conoce, el sistema no se conserva. Pero hay que distinguir entre faltas "masivas" y "marginales".

In [ ]:
ejec_clientes = set(clientes['cod_ejec_bco'].dropna().unique())
ejec_ecas     = set(ecas['cod_ejec_bco'].dropna().unique())
gte_planta    = set(planta['cod_gte_inv'].dropna().unique())
gte_ecas      = set(ecas['cod_gte_inv'].dropna().unique())

diag = pd.DataFrame([
    ['Ejecutivos en clientes', len(ejec_clientes)],
    ['Ejecutivos en ecas (con gerente)', len(ejec_ecas)],
    ['Ejec. huérfanos (clientes − ecas)', len(ejec_clientes - ejec_ecas)],
    ['Ejec. solo en ecas (sin clientes)', len(ejec_ecas - ejec_clientes)],
    ['Ejec. en ambas tablas (intersección)', len(ejec_clientes & ejec_ecas)],
    ['Gerentes en planta', len(gte_planta)],
    ['Gerentes en ecas', len(gte_ecas)],
    ['Gerentes en planta no en ecas (sin equipo)', len(gte_planta - gte_ecas)],
    ['Gerentes en ecas no en planta', len(gte_ecas - gte_planta)],
], columns=['concepto', 'n'])
diag

In [ ]:
# Cuantificar impacto en CLIENTES (no solo en ejecutivos)
huerfanos = ejec_clientes - ejec_ecas
asignables = ejec_clientes & ejec_ecas

clientes_huerfanos = clientes[clientes['cod_ejec_bco'].isin(huerfanos)]
clientes_asignables = clientes[clientes['cod_ejec_bco'].isin(asignables)]
clientes_sin_ejec = clientes[clientes['cod_ejec_bco'].isna()]

impacto = pd.DataFrame([
    {
        'grupo': 'Asignables (ejec en ambas)',
        'clientes': len(clientes_asignables),
        'pct': len(clientes_asignables) / len(clientes) * 100,
        'A': (clientes_asignables['marca_mac_inv'] == 'A').sum(),
        'B': (clientes_asignables['marca_mac_inv'] == 'B').sum(),
        'C': (clientes_asignables['marca_mac_inv'] == 'C').sum(),
    },
    {
        'grupo': 'Huérfanos (ejec no en ecas)',
        'clientes': len(clientes_huerfanos),
        'pct': len(clientes_huerfanos) / len(clientes) * 100,
        'A': (clientes_huerfanos['marca_mac_inv'] == 'A').sum(),
        'B': (clientes_huerfanos['marca_mac_inv'] == 'B').sum(),
        'C': (clientes_huerfanos['marca_mac_inv'] == 'C').sum(),
    },
    {
        'grupo': 'Sin ejecutivo',
        'clientes': len(clientes_sin_ejec),
        'pct': len(clientes_sin_ejec) / len(clientes) * 100,
        'A': (clientes_sin_ejec['marca_mac_inv'] == 'A').sum(),
        'B': (clientes_sin_ejec['marca_mac_inv'] == 'B').sum(),
        'C': (clientes_sin_ejec['marca_mac_inv'] == 'C').sum(),
    },
])
impacto['pct'] = impacto['pct'].round(1)
impacto

In [ ]:
# Visualización del impacto: barras apiladas A/B/C por grupo
impacto_long = impacto.melt(id_vars=['grupo'], value_vars=['A', 'B', 'C'],
                             var_name='categoria', value_name='n_clientes')
fig = px.bar(
    impacto_long, x='grupo', y='n_clientes', color='categoria',
    category_orders={'categoria': ['A', 'B', 'C']},
    title='Impacto de la integridad referencial sobre la población',
    color_discrete_map={'A': BANCOLOMBIA_COLORS['negro'],
                        'B': BANCOLOMBIA_COLORS['amarillo'],
                        'C': BANCOLOMBIA_COLORS['gris']},
    text='n_clientes',
)
fig.update_traces(textposition='inside')
fig.update_layout(xaxis_title=None, yaxis_title='# clientes', legend_title='Categoría')
fig.show()

### 📏 Hallazgos críticos sobre integridad

**🚨 Resultado importante:** **633 ejecutivos huérfanos** (de 1,003 → 63%). El número parece alarmante, **pero lo que importa es cuántos clientes arrastran**.

**Interpretación de los conjuntos:**

1. **Ejecutivos huérfanos (633)**: aparecen en `clientes` pero no en `ecas`. **Probablemente NO son ejecutivos de inversión**, sino ejecutivos de otras bancas (ej. PYME, Empresas) que también gestionan estos clientes en otros productos. Para el modelo solo importan los 392 que sí están en `ecas` con un Gerente de Inversión asociado. Los clientes "huérfanos" pasan al denominador `y` pero no al `x`.

2. **22 ejecutivos solo en `ecas`**: están en la planta pero no tienen clientes asignados actualmente — son **capacidad ociosa** (ejecutivos en transición o nuevos). No los necesitamos para el modelo.

3. **1 gerente en `planta` no aparece en `ecas`**: hay 50 gerentes en planta pero solo 49 con equipo. Ese gerente está en la planta pero **no tiene ejecutivos asignados** → es capacidad disponible no aprovechada actualmente. **Verificar** si su `cod_region_gte_inv` permite asignarle ejecutivos del modelo.

**Lo verdaderamente importante**: el % de A y B en el grupo "asignables" determina el techo real de la métrica. Si los huérfanos arrastran muchos A, el modelo no podrá superar ese techo aunque sea óptimo dentro de los asignables.

---
## 7. Implicaciones concretas para el modelo de optimización

### 🎯 Tamaño efectivo del problema (con datos reales)

| Concepto | Total | Asignable | % asignable |
|---|---|---|---|
| **Clientes** | 34,145 | **27,115** | **79.4%** |
| **Clientes A** | 5,495 | 3,916 | 71.3% |
| **Clientes B** | 12,727 | 10,000 | 78.6% |
| **Clientes C** | 15,923 | 13,199 | 82.9% |
| **Ejecutivos** | 1,003 (en clientes) | **392** (intersección con ecas) | 39.1% |
| **Gerentes** | 50 (en planta) | **49** (con equipo en ecas) | 98% |
| **Variables MILP brutas** | 392 × 49 = 19,208 | con filtro de zona ~3-5k | |

### 🚨 Hallazgo crítico: techo real de la métrica

| Escenario | Cálculo | Valor |
|---|---|---|
| **Techo absoluto teórico** | (5,495 + 12,727) / 34,145 | **53.4%** |
| **Techo real considerando huérfanos** | (3,916 + 10,000) / 34,145 | **40.8%** |
| **Pérdida estructural por huérfanos** | 53.4% − 40.8% | **12.6 puntos** |

> **Lectura**: aunque tuviéramos un modelo perfecto que asignara cada A y B asignable a un gerente, **la métrica jamás superará 40.8%** por la limitación estructural de las llaves. Los 4,306 clientes A+B huérfanos (1,579 A + 2,727 B) son **no asignables vía Gerentes de Inversión** porque sus ejecutivos no están en la jerarquía `ecas`.

### 📋 Pesos del modelo (confirmados)

- $w_A = 1000, \; w_B = 100, \; w_C = 1$
- $\alpha = 0.1$ (peso del score como tie-breaker)
- **Cuidado con el score negativo**: los valores < 0 restan valor (cliente comercialmente indeseable resta), lo cual es coherente. No normalizar a [0,1] antes de usarlo.

### ⚠️ Riesgos detectados con cuantificación

| Riesgo | Magnitud | Mitigación |
|---|---|---|
| **Score con valores negativos** | min = −0.03 | Usar tal cual — coherente comercialmente |
| **Distribución de carga sesgada** | max=284, mediana=8 | Verificar que el "super-ejecutivo" tenga gerente con capacidad |
| **1 gerente sin equipo en ecas** | 1 de 50 | Capacidad no utilizable a menos de tener ejecutivos compatibles en su zona |
| **633 ejecutivos huérfanos** | arrastran 7,030 clientes (20.6%) | Reportar como "no asignables estructurales" en el documento |
| **Clientes A perdidos** | 1,579 (28.7% de los A totales) | Aceptar como límite estructural |

### 🔄 Conexión con notebook 02

En `02_eda_capacidad_y_tiempos.ipynb`:
- Calculamos $\tau_c$ (tiempo demandado por cliente) y $T_g$ (capacidad por gerente).
- Comparamos demanda vs capacidad en el conjunto **asignable** (27,115 clientes).
- Si la capacidad no alcanza, el techo real de 40.8% baja aún más.

### 🎓 Cierre del notebook

Este notebook fija **dos límites del modelo** que ya conocemos antes de optimizar:
1. **Techo estructural** (huérfanos): la métrica nunca supera 40.8%.
2. **Tamaño manejable** (392 ejecutivos × 49 gerentes con filtro de zona): MILP exacto es viable.

Estos números van directo al **documento final** como contexto para interpretar el resultado del modelo.